# Evaluate submission_5 with unchanged Gemma weights

Compare `submission_4` with `submission_5` on identical tasks, three-agent structure, prompts, model and four-minute agent allowance. The candidate raises the command/grading timeout from 60 to 300 seconds and requests thinking disabled on all three agents. Controls change each setting separately. Their effective runtime behavior and task performance must be verified in Kaggle.

Attach the competition data, Gemma model, and Gemma 4 Developer Agent Wheelhouse. Enable GPU acceleration and run cells in order. The default two tasks provide a smoke test. Increase task coverage for comparisons, keep a held-out set for later tuning, and repeat runs to assess generation variation. This notebook evaluates and packages; it performs no training.

## 1. Choose data, candidate, tasks and variants

Set `SUBMISSION_SOURCE_DIR` only for your mounted candidate. Explicit `TASK_IDS` preserve a fixed validation set across revisions. Runs use unique directories and preserve earlier artifacts.

In [ ]:
from pathlib import Path

DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
MODEL_PATH = Path('/kaggle/input/models/google/gemma-4/other/gemma-4-31b-it-qat-w4a16-ct/2')
# A mounted candidate must preserve this comparison's permitted changes.
# None uses the curated candidate embedded when this notebook was generated.
SUBMISSION_SOURCE_DIR = None
BENCHMARK_VARIANTS = ['baseline', 'timeout_only', 'thinking_off_only', 'candidate']
RELEASE_VARIANT = 'candidate'
TASK_IDS = []  # Explicit IDs take precedence over TASK_LIMIT.
TASK_LIMIT = 2  # Smoke test only; None evaluates all tasks with complete resources.
SANDBOX = 'subprocess'  # Kaggle starter mode. Use 'docker' where Docker/image are available.
WORKING_DIR = Path('/kaggle/working')


## 2. Install the provided offline runtime

This installation cell comes from the starter notebook. The task `wheels/` directory is a separate dependency collection.

In [ ]:
import glob
import importlib
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Configure environment variables for offline vLLM serving and LiteLLM routing
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')

# Remove broken cutlass .pth hooks if present
for pth_pattern in (
    '/usr/local/lib/python*/dist-packages/*cutlass*.pth',
    '/usr/local/lib/python*/site-packages/*cutlass*.pth',
):
    for pth in glob.glob(pth_pattern):
        try:
            os.unlink(pth)
        except OSError:
            pass

# Restore PEP 440 '+cu128' wheel filenames stripped by Kaggle dataset uploads
tmp_whl = Path('/tmp/wheelhouse')
tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower():
        continue
    target_name = (
        w.name.replace('cu128', '+cu128')
        if ('cu128' in w.name and '+' not in w.name)
        else w.name
    )
    target = tmp_whl / target_name
    if not target.exists():
        os.symlink(w, target)

wheels = sorted(str(w) for w in tmp_whl.glob('*.whl'))
print(f'Installing {len(wheels)} wheels from {WHEELHOUSE_DIR}...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels],
    check=True,
)
importlib.invalidate_caches()
print('Wheelhouse installation complete.')


## 3. Materialize runtime controls

`baseline` retains Submission 4. `timeout_only` changes only its timeout to 300 seconds; `thinking_off_only` disables thinking while retaining the 60-second timeout; `candidate` applies both changes and is the default release. Preflight rejects changes to other budgets, sampling, prompts, tools or agent declarations. These controls retain all three agents, with shared task budgets and optional helpers.

In [ ]:
# Curated text files; regenerate this notebook after editing the candidate or tools.
EMBEDDED_SUBMISSION = {'agent.yaml': 'name: swe_coder_agent\nmodel: gemma-4-31b-it-qat-w4a16-ct\ninstruction: !include prompts/system.md\ntools:\n  - run_command\n  - read_file\n  - edit_file\n  - write_file\n  - get_status\n  - submit_patch\n  - agent_tool:\n      config_path: sub_agents/repository_locator.yaml\n      skip_summarization: true\n  - agent_tool:\n      config_path: sub_agents/test_verifier.yaml\n      skip_summarization: true\ngenerate_content_config: !include configs/sampling.yaml\n', 'configs/sampling.yaml': 'temperature: 0.2\ntop_p: 0.95\nmax_output_tokens: 16384\nthinking_config:\n  include_thoughts: false\n', 'eval_config.yaml': "# Command timeout is also used by grading in the inspected harness.\n# Agent time excludes sandbox setup, while the competition's 12 hours includes it.\nevaluation:\n  timeout_seconds: 300\n  max_tool_calls: 40\n  max_time_minutes: 4\n  max_turns: 80\n", 'prompts/system.md': 'Implement the issue in /workspace and call submit_patch to capture the complete implementation patch. You own all source edits, repairs and submission. Use actual tools until the work is finished; a plan or diagnosis is not completion. A final text response can end the task, so submit the patch before giving a completion response. Keep reasoning concise and act on supported findings.\n\nLocate the cause using issue paths, literal errors, named symbols and relevant tests. Use bounded rg or git grep and focused file reads; refine unsuccessful searches. read_file line numbers are integers. After a failed edit match, read the current source before retrying. Once source supports a fix, edit it without repeatedly confirming the same finding. Cover all requested behaviors and edge cases, preserving required signatures, exception types, messages and compatibility. Prefer small uniquely matched edits over whole-file rewrites.\n\nTwo helpers may be available. They have no full history and share your task budget. Each may be called at most once by prompt convention; they have no separate hard allowance. Do not delegate an obvious command or send both helpers the same question. If a helper is unavailable, fails or returns insufficient evidence, continue with your own tools.\n\n- repository_locator_agent: use only when the affected implementation or a relevant test remains unclear. Check get_status first; skip when less than 150 seconds or eight counted calls remain. Supply the issue, known paths/symbols, inspected evidence and one precise localization question. Request affected implementation/test paths and decisive source evidence in at most 120 words. It can search with run_command and read_file. Confirm decisive findings without repeating its entire search.\n- test_verifier_agent: use after a source patch exists when selecting a behavioral check or interpreting a failure needs help. Check get_status first; skip when less than 90 seconds or six counted calls remain. Supply the required behavior, changed paths, relevant diff/source ranges, proposed test target and commands/results already observed. Request one focused execution and a concise evidence report. Reuse a verification diff if already available; identify any omitted hunks. It tests and diagnoses; you make repairs. If unavailable or unnecessary, run the check yourself.\n\nUse targeted existing tests or a small assertion-based reproduction, normally capped with timeout 20s when available. Inspect actual output. Repair demonstrated implementation defects and recheck when time permits. Collection/dependency failures and timeouts are inconclusive; choose a smaller available check rather than installing packages or repeating a failing command. Helpers may run shell commands but are instructed to leave implementation, protected tests and environment unchanged. Put scratch reproductions under /tmp. Do not change protected tests, runner configuration or harness files to obtain a pass.\n\nReserve the final 30 seconds for repair, final inspection and patch capture. Near exhaustion, stop delegation and broad exploration. Inspect changed paths, a bounded diff and git diff --check; include required new implementation files with git add -N so they appear in the diff. Address each requested behavior as far as evidence permits. Remove only your scratch artifacts. A passing local check is evidence, not proof of hidden-test success. Call submit_patch as the final tool action even when verification is incomplete; never discard a supported implementation just because a helper was skipped.\n', 'sub_agents/locator_resources/instruction.md': "Answer one localization question for the coder using the supplied issue and repository evidence. You have no full history. Use run_command for bounded rg, git grep, file listings or source inspection, and read_file for precise ranges. Do not edit source, write workspace files, install dependencies, run tests or submit patches. Shell access is for inspection. Treat missing issue details as uncertainty rather than inventing requirements.\n\nStart from concrete paths, symbols and literal error text in the request. Normally use one or two tool calls and stop as soon as the location is supported. Refine a failed search instead of repeating it; bound output and respect truncation. Avoid broad repository surveys. These call targets are behavioral instructions; your inference and tools share the coder's task allowance.\n\nReturn at most 120 words:\n- Implementation: path, symbol/line range and decisive current-source evidence.\n- Verification location: one relevant existing test path/node, or explicitly unknown. Do not run it.\n- Suggested change: the smallest supported correction and relevant callers/edge cases.\n- Uncertainty: missing evidence or one next focused check, if needed.\n\nDo not return a full plan, copied files, speculative alternatives or claims that a fix passes tests. Return your findings to the coder immediately; the coder edits, verifies and submits.\n", 'sub_agents/locator_resources/sampling.yaml': 'temperature: 0.2\ntop_p: 0.95\nmax_output_tokens: 2048\nthinking_config:\n  include_thoughts: false\n', 'sub_agents/repository_locator.yaml': 'name: repository_locator_agent\ndescription: Finds implementation and existing test locations for one unresolved issue using bounded shell search and source reads. Returns concise evidence; does not edit or submit.\nmodel: gemma-4-31b-it-qat-w4a16-ct\ninstruction: !include locator_resources/instruction.md\ninclude_contents: none\ntools:\n  - run_command\n  - read_file\ngenerate_content_config: !include locator_resources/sampling.yaml\n', 'sub_agents/test_verifier.yaml': 'name: test_verifier_agent\ndescription: Executes one focused behavioral check for an existing patch and distinguishes assertions, collection/dependency errors and timeouts. Reports evidence and a minimal repair suggestion; does not edit or submit.\nmodel: gemma-4-31b-it-qat-w4a16-ct\ninstruction: !include verifier_resources/instruction.md\ninclude_contents: none\ntools:\n  - run_command\n  - read_file\ngenerate_content_config: !include verifier_resources/sampling.yaml\n', 'sub_agents/verifier_resources/instruction.md': "Verify one requested behavior of the coder's current patch using the supplied issue, diff/source ranges, changed paths and prior command results. You have no full history. You can run_command and read_file; you cannot use editing or submission tools. Do not modify implementation, existing tests, runner configuration, harness files, git state or dependencies. Do not reset the workspace. A necessary small reproduction may be written only under /tmp and must check observable behavior with assertions.\n\nPrefer an existing focused test node or a short behavioral reproduction. Normally execute one command, using timeout 20s when available; use at most one focused source read to select or interpret it. Do not rerun a check already supplied as successful unless the patch has changed or new evidence warrants it. If a prior check failed, choose the smallest useful diagnosis instead of blindly repeating it. Do not run the full suite, install packages or perform broad exploration. Your tools and inference share the coder's task budget; these targets are prompt conventions, not independent quotas.\n\nReturn at most 150 words:\n- Check: exact executed command or explicitly not run, with the behavior checked.\n- Outcome: pass, assertion failure, collection/dependency failure, timeout, tool error or inconclusive. Quote only decisive output and the exit status when visible.\n- Repair evidence: a supported source path/cause and minimal correction, or explicitly unknown. Suggest; never edit.\n- Coverage gap: any important requested behavior this check did not cover.\n\nA command timeout, collection error or no collected tests is not a pass. A successful check establishes only its tested behavior. Do not claim hidden tests pass or infer errors are unrelated without evidence. Return immediately so the coder can repair and submit; no repeated helper loop.\n", 'sub_agents/verifier_resources/sampling.yaml': 'temperature: 0.2\ntop_p: 0.95\nmax_output_tokens: 2048\nthinking_config:\n  include_thoughts: false\n'}
EMBEDDED_BASELINE = {'agent.yaml': 'name: swe_coder_agent\nmodel: gemma-4-31b-it-qat-w4a16-ct\ninstruction: !include prompts/system.md\ntools:\n  - run_command\n  - read_file\n  - edit_file\n  - write_file\n  - get_status\n  - submit_patch\n  - agent_tool:\n      config_path: sub_agents/repository_locator.yaml\n      skip_summarization: true\n  - agent_tool:\n      config_path: sub_agents/test_verifier.yaml\n      skip_summarization: true\ngenerate_content_config: !include configs/sampling.yaml\n', 'configs/sampling.yaml': 'temperature: 0.2\ntop_p: 0.95\nmax_output_tokens: 16384\nthinking_config:\n  thinking_budget: 4096\n  include_thoughts: true\n', 'eval_config.yaml': "# Initial evaluation budgets; validate total inference time before submission.\n# Agent time excludes sandbox setup, while the competition's 12 hours includes it.\nevaluation:\n  timeout_seconds: 60\n  max_tool_calls: 40\n  max_time_minutes: 4\n  max_turns: 80\n", 'prompts/system.md': 'Implement the issue in /workspace and call submit_patch to capture the complete implementation patch. You own all source edits, repairs and submission. Use actual tools until the work is finished; a plan or diagnosis is not completion. A final text response can end the task, so submit the patch before giving a completion response. Keep reasoning concise and act on supported findings.\n\nLocate the cause using issue paths, literal errors, named symbols and relevant tests. Use bounded rg or git grep and focused file reads; refine unsuccessful searches. read_file line numbers are integers. After a failed edit match, read the current source before retrying. Once source supports a fix, edit it without repeatedly confirming the same finding. Cover all requested behaviors and edge cases, preserving required signatures, exception types, messages and compatibility. Prefer small uniquely matched edits over whole-file rewrites.\n\nTwo helpers may be available. They have no full history and share your task budget. Each may be called at most once by prompt convention; they have no separate hard allowance. Do not delegate an obvious command or send both helpers the same question. If a helper is unavailable, fails or returns insufficient evidence, continue with your own tools.\n\n- repository_locator_agent: use only when the affected implementation or a relevant test remains unclear. Check get_status first; skip when less than 150 seconds or eight counted calls remain. Supply the issue, known paths/symbols, inspected evidence and one precise localization question. Request affected implementation/test paths and decisive source evidence in at most 120 words. It can search with run_command and read_file. Confirm decisive findings without repeating its entire search.\n- test_verifier_agent: use after a source patch exists when selecting a behavioral check or interpreting a failure needs help. Check get_status first; skip when less than 90 seconds or six counted calls remain. Supply the required behavior, changed paths, relevant diff/source ranges, proposed test target and commands/results already observed. Request one focused execution and a concise evidence report. Reuse a verification diff if already available; identify any omitted hunks. It tests and diagnoses; you make repairs. If unavailable or unnecessary, run the check yourself.\n\nUse targeted existing tests or a small assertion-based reproduction, normally capped with timeout 20s when available. Inspect actual output. Repair demonstrated implementation defects and recheck when time permits. Collection/dependency failures and timeouts are inconclusive; choose a smaller available check rather than installing packages or repeating a failing command. Helpers may run shell commands but are instructed to leave implementation, protected tests and environment unchanged. Put scratch reproductions under /tmp. Do not change protected tests, runner configuration or harness files to obtain a pass.\n\nReserve the final 30 seconds for repair, final inspection and patch capture. Near exhaustion, stop delegation and broad exploration. Inspect changed paths, a bounded diff and git diff --check; include required new implementation files with git add -N so they appear in the diff. Address each requested behavior as far as evidence permits. Remove only your scratch artifacts. A passing local check is evidence, not proof of hidden-test success. Call submit_patch as the final tool action even when verification is incomplete; never discard a supported implementation just because a helper was skipped.\n', 'sub_agents/locator_resources/instruction.md': "Answer one localization question for the coder using the supplied issue and repository evidence. You have no full history. Use run_command for bounded rg, git grep, file listings or source inspection, and read_file for precise ranges. Do not edit source, write workspace files, install dependencies, run tests or submit patches. Shell access is for inspection. Treat missing issue details as uncertainty rather than inventing requirements.\n\nStart from concrete paths, symbols and literal error text in the request. Normally use one or two tool calls and stop as soon as the location is supported. Refine a failed search instead of repeating it; bound output and respect truncation. Avoid broad repository surveys. These call targets are behavioral instructions; your inference and tools share the coder's task allowance.\n\nReturn at most 120 words:\n- Implementation: path, symbol/line range and decisive current-source evidence.\n- Verification location: one relevant existing test path/node, or explicitly unknown. Do not run it.\n- Suggested change: the smallest supported correction and relevant callers/edge cases.\n- Uncertainty: missing evidence or one next focused check, if needed.\n\nDo not return a full plan, copied files, speculative alternatives or claims that a fix passes tests. Return your findings to the coder immediately; the coder edits, verifies and submits.\n", 'sub_agents/locator_resources/sampling.yaml': 'temperature: 0.2\ntop_p: 0.95\nmax_output_tokens: 2048\nthinking_config:\n  thinking_budget: 1024\n  include_thoughts: true\n', 'sub_agents/repository_locator.yaml': 'name: repository_locator_agent\ndescription: Finds implementation and existing test locations for one unresolved issue using bounded shell search and source reads. Returns concise evidence; does not edit or submit.\nmodel: gemma-4-31b-it-qat-w4a16-ct\ninstruction: !include locator_resources/instruction.md\ninclude_contents: none\ntools:\n  - run_command\n  - read_file\ngenerate_content_config: !include locator_resources/sampling.yaml\n', 'sub_agents/test_verifier.yaml': 'name: test_verifier_agent\ndescription: Executes one focused behavioral check for an existing patch and distinguishes assertions, collection/dependency errors and timeouts. Reports evidence and a minimal repair suggestion; does not edit or submit.\nmodel: gemma-4-31b-it-qat-w4a16-ct\ninstruction: !include verifier_resources/instruction.md\ninclude_contents: none\ntools:\n  - run_command\n  - read_file\ngenerate_content_config: !include verifier_resources/sampling.yaml\n', 'sub_agents/verifier_resources/instruction.md': "Verify one requested behavior of the coder's current patch using the supplied issue, diff/source ranges, changed paths and prior command results. You have no full history. You can run_command and read_file; you cannot use editing or submission tools. Do not modify implementation, existing tests, runner configuration, harness files, git state or dependencies. Do not reset the workspace. A necessary small reproduction may be written only under /tmp and must check observable behavior with assertions.\n\nPrefer an existing focused test node or a short behavioral reproduction. Normally execute one command, using timeout 20s when available; use at most one focused source read to select or interpret it. Do not rerun a check already supplied as successful unless the patch has changed or new evidence warrants it. If a prior check failed, choose the smallest useful diagnosis instead of blindly repeating it. Do not run the full suite, install packages or perform broad exploration. Your tools and inference share the coder's task budget; these targets are prompt conventions, not independent quotas.\n\nReturn at most 150 words:\n- Check: exact executed command or explicitly not run, with the behavior checked.\n- Outcome: pass, assertion failure, collection/dependency failure, timeout, tool error or inconclusive. Quote only decisive output and the exit status when visible.\n- Repair evidence: a supported source path/cause and minimal correction, or explicitly unknown. Suggest; never edit.\n- Coverage gap: any important requested behavior this check did not cover.\n\nA command timeout, collection error or no collected tests is not a pass. A successful check establishes only its tested behavior. Do not claim hidden tests pass or infer errors are unrelated without evidence. Return immediately so the coder can repair and submit; no repeated helper loop.\n", 'sub_agents/verifier_resources/sampling.yaml': 'temperature: 0.2\ntop_p: 0.95\nmax_output_tokens: 2048\nthinking_config:\n  thinking_budget: 1024\n  include_thoughts: true\n'}
EMBEDDED_TOOL_SOURCE = '"""Local-only validation, packaging, and evidence-based evaluation diagnostics.\n\nBuild packages the dependency closure of agent.yaml and eval_config.yaml, plus\nwhole referenced skill/adapter directories. Unreferenced documentation is omitted.\nThis checks the documented submission subset; Kaggle\'s compiler is authoritative.\nReport never treats task duration or cumulative tokens as agent time/context size.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport json\nimport math\nimport re\nimport sys\nimport zipfile\nfrom collections import Counter\nfrom datetime import datetime\nfrom pathlib import Path, PureWindowsPath\nfrom typing import Any\n\nimport yaml\n\nMODEL = "gemma-4-31b-it-qat-w4a16-ct"\nCORE_TOOLS = frozenset({\n    "run_command", "read_file", "edit_file", "write_file", "get_status",\n    "submit_patch", "get_code_neighbors", "search_similar_code", "get_code_subgraph",\n})\nFREE_TOOLS = frozenset({"get_status", "submit_patch"})\nSKILL_TOOLS = frozenset({"load_skill_resource", "run_skill_script"})\nALLOWED_EXTENSIONS = frozenset({".yaml", ".yml", ".md", ".txt", ".py", ".json", ".safetensors"})\nMAX_BYTES = 3 * 1024 ** 3\nMAX_YAML_BYTES = 50 * 1024 ** 2\n\n\nclass ValidationError(ValueError):\n    pass\n\n\ndef _inside(path: Path, root: Path) -> bool:\n    try:\n        path.relative_to(root)\n        return True\n    except ValueError:\n        return False\n\n\nclass SubmissionValidator:\n    """Resolve dependencies without importing competitor Python or executing YAML."""\n\n    def __init__(self, directory: str | Path):\n        requested = Path(directory).absolute()\n        if requested.is_symlink():\n            raise ValidationError("Submission directory must not be a symlink")\n        self.root = requested.resolve()\n        self.files: set[Path] = set()\n        self.agent_paths: set[Path] = set()\n        self.adapter_names: set[str] = set()\n        self.agent_tool_names: set[str] = set()\n        self.agent_stack: set[Path] = set()\n        self.agents = 0\n        self.warnings: list[str] = []\n\n    def resolve(self, ref: Any, parent: Path, *, directory: bool = False) -> Path:\n        if not isinstance(ref, str) or not ref or "\\x00" in ref:\n            raise ValidationError(f"Invalid path reference: {ref!r}")\n        windows = PureWindowsPath(ref)\n        if Path(ref).is_absolute() or windows.is_absolute() or windows.drive or ref.startswith(("/", "\\\\")):\n            raise ValidationError(f"Absolute path is forbidden: {ref}")\n        # Normalize separators so Windows-style traversal is also detected on Linux.\n        raw = parent / Path(ref.replace("\\\\", "/"))\n        resolved = raw.resolve()\n        if not _inside(resolved, self.root):\n            raise ValidationError(f"Path escapes submission root: {ref}")\n        for part in [raw, *raw.parents]:\n            if part.is_symlink():\n                raise ValidationError(f"Symlink is forbidden: {ref}")\n            if part == self.root:\n                break\n        if directory and not resolved.is_dir():\n            raise ValidationError(f"Missing directory: {ref}")\n        if not directory and not resolved.is_file():\n            raise ValidationError(f"Missing file: {ref}")\n        return resolved\n\n    def load_yaml(self, path: Path, stack: tuple[Path, ...] = ()) -> Any:\n        if path in stack:\n            raise ValidationError(f"Include cycle: {path.relative_to(self.root)}")\n        if len(stack) > 10:\n            raise ValidationError("Include nesting exceeds 10")\n        if path.stat().st_size > MAX_YAML_BYTES:\n            raise ValidationError(f"YAML exceeds 50 MiB: {path.name}")\n        self.files.add(path)\n        owner = self\n\n        class Loader(yaml.SafeLoader):\n            pass\n\n        def include(loader: Loader, node: yaml.Node) -> Any:\n            target = owner.resolve(loader.construct_scalar(node), path.parent)\n            owner.files.add(target)\n            if target.suffix.lower() in {".yaml", ".yml"}:\n                return owner.load_yaml(target, (*stack, path))\n            if target.suffix.lower() not in {".md", ".txt"}:\n                raise ValidationError(f"Unsupported include format: {target.name}")\n            return target.read_text(encoding="utf-8")\n\n        Loader.add_constructor("!include", include)\n        try:\n            return yaml.load(path.read_text(encoding="utf-8"), Loader=Loader)\n        except (yaml.YAMLError, UnicodeError, RecursionError) as exc:\n            raise ValidationError(f"Invalid YAML in {path.name}: {exc}") from exc\n\n    def collect_directory(self, directory: Path) -> None:\n        for path in directory.rglob("*"):\n            if path.is_symlink():\n                raise ValidationError(f"Symlink is forbidden: {path.relative_to(self.root)}")\n            if path.is_file():\n                self.files.add(path)\n\n    def generation_config(self, config: Any) -> None:\n        if config is None:\n            return\n        if not isinstance(config, dict):\n            raise ValidationError("generate_content_config must be a mapping")\n        permitted = {\n            "temperature", "top_p", "top_k", "max_output_tokens", "presence_penalty",\n            "frequency_penalty", "stop_sequences", "response_mime_type", "seed", "thinking_config",\n        }\n        extra = set(config) - permitted\n        if extra:\n            raise ValidationError(f"Unsupported generation fields: {sorted(extra)}")\n        for key, lower, upper in [("max_output_tokens", 1, 32768), ("top_p", 0, 1), ("temperature", 0, None), ("top_k", 1, None)]:\n            if key in config:\n                value = config[key]\n                if not isinstance(value, (int, float)) or isinstance(value, bool) or not math.isfinite(value) or value < lower or (upper is not None and value > upper):\n                    raise ValidationError(f"Invalid generation setting: {key}")\n                if key in {"max_output_tokens", "top_k"} and not isinstance(value, int):\n                    raise ValidationError(f"{key} must be an integer")\n        thinking = config.get("thinking_config")\n        if thinking is not None:\n            if not isinstance(thinking, dict) or set(thinking) - {"thinking_budget", "thinking_level", "include_thoughts"}:\n                raise ValidationError("Unsupported thinking_config")\n            if "thinking_budget" in thinking:\n                value = thinking["thinking_budget"]\n                if not isinstance(value, int) or isinstance(value, bool) or not 0 <= value <= 32768:\n                    raise ValidationError("thinking_budget must be an integer in 0..32768")\n            if "include_thoughts" in thinking and not isinstance(thinking["include_thoughts"], bool):\n                raise ValidationError("include_thoughts must be boolean")\n            if "thinking_level" in thinking:\n                if str(thinking["thinking_level"]).upper() not in {"MINIMAL", "LOW", "MEDIUM", "HIGH", "NONE"}:\n                    raise ValidationError("Invalid thinking_level")\n                self.warnings.append("thinking_level can map to unsupported reasoning_effort; prefer thinking_budget")\n\n    def agent(self, config: Any, path: Path, depth: int = 0, *, as_tool: bool = False) -> None:\n        if depth > 50 or self.agents >= 500:\n            raise ValidationError("Agent tree exceeds documented depth/count limits")\n        if not isinstance(config, dict):\n            raise ValidationError(f"Agent config must be a mapping: {path.name}")\n        self.agents += 1\n        kind = config.get("agent_class", "LlmAgent")\n        if not isinstance(kind, str) or kind not in {"LlmAgent", "SequentialAgent", "ParallelAgent", "LoopAgent"}:\n            raise ValidationError(f"Unsupported agent_class: {kind}")\n        name = config.get("name")\n        if not isinstance(name, str) or not name:\n            raise ValidationError(f"Missing agent name: {path.name}")\n        common_fields = {"name", "description", "agent_class", "sub_agents"}\n        llm_fields = {\n            "model", "adapter", "instruction", "global_instruction", "tools", "skills",\n            "output_key", "include_contents", "disallow_transfer_to_parent",\n            "disallow_transfer_to_peers", "generate_content_config",\n        }\n        permitted = common_fields | (llm_fields if kind == "LlmAgent" else {"max_iterations"} if kind == "LoopAgent" else set())\n        if set(config) - permitted:\n            raise ValidationError(f"Unsupported agent fields in local documented subset: {sorted(set(config) - permitted)}")\n        if as_tool:\n            self.agent_tool_names.add(name)\n        if kind == "LlmAgent":\n            if config.get("model") != MODEL:\n                raise ValidationError(f"Agent {name} must declare model {MODEL}")\n            if not isinstance(config.get("instruction", ""), str):\n                raise ValidationError(f"instruction must be text: {name}")\n            if len(config.get("instruction", "")) > 1_000_000:\n                raise ValidationError(f"instruction exceeds 1,000,000 characters: {name}")\n            if config.get("include_contents", "default") not in {"default", "none"}:\n                raise ValidationError(f"Invalid include_contents: {name}")\n            self.generation_config(config.get("generate_content_config"))\n            adapter = config.get("adapter")\n            if adapter is not None:\n                if not isinstance(adapter, str) or "/" in adapter or "\\\\" in adapter or adapter in {".", ".."}:\n                    raise ValidationError("adapter must be a directory name under adapters/")\n                directory = self.resolve(f"adapters/{adapter}", self.root, directory=True)\n                self.resolve("adapter_config.json", directory)\n                self.resolve("adapter_model.safetensors", directory)\n                self.adapter_names.add(adapter)\n                self.collect_directory(directory)\n            skills = config.get("skills") or []\n            if not isinstance(skills, list):\n                raise ValidationError("skills must be a list")\n            for ref in skills:\n                directory = self.resolve(ref, self.root, directory=True)\n                manifest = self.resolve("SKILL.md", directory)\n                text = manifest.read_text(encoding="utf-8")\n                match = re.match(r"\\A---\\s*\\n(.*?)\\n---(?:\\s*\\n|$)", text, re.S)\n                metadata = yaml.safe_load(match.group(1)) if match else None\n                if not isinstance(metadata, dict) or not isinstance(metadata.get("name"), str) or not metadata["name"]:\n                    raise ValidationError(f"Skill manifest needs YAML frontmatter with name: {ref}")\n                self.collect_directory(directory)\n            tools = config.get("tools", [])\n            if not isinstance(tools, list):\n                raise ValidationError("tools must be a list")\n            for tool in tools:\n                if isinstance(tool, str):\n                    if tool not in CORE_TOOLS and not (skills and tool in SKILL_TOOLS):\n                        raise ValidationError(f"Unsupported tool {tool!r} in {name}")\n                elif isinstance(tool, dict) and "agent_tool" in tool:\n                    if set(tool) != {"agent_tool"} or not isinstance(tool["agent_tool"], dict):\n                        raise ValidationError("agent_tool must be its own mapping")\n                    spec = tool["agent_tool"]\n                    if set(spec) - {"config_path", "skip_summarization"}:\n                        raise ValidationError("Unsupported agent_tool fields")\n                    if "skip_summarization" in spec and not isinstance(spec["skip_summarization"], bool):\n                        raise ValidationError("skip_summarization must be boolean")\n                    self.agent_reference(spec, path, depth, as_tool=True)\n                elif isinstance(tool, dict) and "name" in tool:\n                    self.agent(tool, path, depth + 1, as_tool=True)\n                else:\n                    raise ValidationError(f"Invalid tool declaration in {name}")\n        if kind == "LoopAgent":\n            iterations = config.get("max_iterations", 500)\n            if not isinstance(iterations, int) or isinstance(iterations, bool) or not 1 <= iterations <= 500:\n                raise ValidationError("max_iterations must be in 1..500")\n        children = config.get("sub_agents", [])\n        if not isinstance(children, list):\n            raise ValidationError("sub_agents must be a list")\n        for child in children:\n            if isinstance(child, dict) and "config_path" in child:\n                self.agent_reference(child, path, depth)\n            elif isinstance(child, dict):\n                self.agent(child, path, depth + 1)\n            else:\n                raise ValidationError("Invalid sub_agents declaration")\n\n    def agent_reference(self, spec: dict, parent: Path, depth: int, *, as_tool: bool = False) -> None:\n        target = self.resolve(spec.get("config_path"), self.root)\n        if target in self.agent_stack:\n            raise ValidationError(f"Agent reference cycle: {target.name}")\n        self.agent_stack.add(target)\n        self.agent_paths.add(target)\n        try:\n            self.agent(self.load_yaml(target), target, depth + 1, as_tool=as_tool)\n        finally:\n            self.agent_stack.remove(target)\n\n    def validate(self) -> dict:\n        if not self.root.is_dir():\n            raise ValidationError(f"Missing submission directory: {self.root}")\n        all_files = []\n        for path in self.root.rglob("*"):\n            if path.is_symlink():\n                raise ValidationError(f"Symlink is forbidden: {path.relative_to(self.root)}")\n            if path.is_file():\n                all_files.append(path)\n                if path.suffix.lower() not in ALLOWED_EXTENSIONS:\n                    raise ValidationError(f"Forbidden file extension: {path.relative_to(self.root)}")\n        if len(all_files) > 10_000:\n            raise ValidationError("Submission exceeds 10,000 files")\n        # Runtime preflight also scans standalone agent YAML files, even if build\n        # would omit them. Inspect declarations without resolving unused includes.\n        class DeclarationLoader(yaml.SafeLoader):\n            pass\n        DeclarationLoader.add_constructor("!include", lambda loader, node: loader.construct_scalar(node))\n        for path in all_files:\n            if path.suffix.lower() not in {".yaml", ".yml"}:\n                continue\n            if path.stat().st_size > MAX_YAML_BYTES:\n                raise ValidationError(f"YAML exceeds 50 MiB: {path.name}")\n            try:\n                declaration = yaml.load(path.read_text(encoding="utf-8"), Loader=DeclarationLoader)\n            except (yaml.YAMLError, UnicodeError, RecursionError) as exc:\n                raise ValidationError(f"Invalid source YAML: {path.name}: {exc}") from exc\n            if isinstance(declaration, dict) and "name" in declaration and "model" in declaration and declaration["model"] != MODEL:\n                raise ValidationError(f"Standalone agent {path.name} declares a different model")\n        root_yaml = self.resolve("agent.yaml", self.root)\n        self.agent_stack.add(root_yaml)\n        self.agent(self.load_yaml(root_yaml), root_yaml)\n        self.agent_stack.remove(root_yaml)\n        evaluation = None\n        eval_path = self.root / "eval_config.yaml"\n        if eval_path.exists():\n            evaluation = self.load_yaml(eval_path)\n            if not isinstance(evaluation, dict) or not isinstance(evaluation.get("evaluation"), dict):\n                raise ValidationError("eval_config.yaml needs an evaluation mapping")\n            allowed = {"timeout_seconds", "max_tool_calls", "max_time_minutes", "max_turns"}\n            for key, value in evaluation["evaluation"].items():\n                if key not in allowed or not isinstance(value, (int, float)) or isinstance(value, bool) or not math.isfinite(value) or value <= 0:\n                    raise ValidationError(f"Invalid evaluation setting: {key}")\n                if key != "max_time_minutes" and not isinstance(value, int):\n                    raise ValidationError(f"{key} must be an integer")\n        stale = [p.relative_to(self.root).as_posix() for p in all_files if p.relative_to(self.root).parts[0] == "adapters" and p not in self.files]\n        if stale:\n            raise ValidationError("Unreferenced adapter assets: " + ", ".join(sorted(stale)))\n        loose_weights = [p for p in all_files if p.suffix.lower() == ".safetensors" and p not in self.files]\n        if loose_weights:\n            raise ValidationError("Unreferenced weight assets are not allowed")\n        total = sum(p.stat().st_size for p in self.files)\n        if total >= MAX_BYTES:\n            raise ValidationError("Unpacked package must be smaller than 3 GiB")\n        packaged = sorted(p.relative_to(self.root).as_posix() for p in self.files)\n        ignored = sorted(p.relative_to(self.root).as_posix() for p in all_files if p not in self.files)\n        if ignored:\n            self.warnings.append("Build omits unreferenced files; inspect omitted_files before packaging")\n        return {\n            "valid": True, "submission_dir": str(self.root), "model": MODEL,\n            "agents": self.agents, "adapters": sorted(self.adapter_names),\n            "agent_tool_names": sorted(self.agent_tool_names), "package_bytes": total,\n            "package_files": packaged, "omitted_files": ignored,\n            "evaluation": evaluation.get("evaluation") if evaluation else None,\n            "warnings": self.warnings,\n            "scope": "Documented local checks; the official runtime compiler remains authoritative",\n        }\n\n\ndef validate_submission(directory: str | Path) -> dict:\n    return SubmissionValidator(directory).validate()\n\n\ndef build_submission(directory: str | Path, output: str | Path) -> dict:\n    validator = SubmissionValidator(directory)\n    manifest = validator.validate()\n    destination = Path(output).absolute().resolve()\n    if _inside(destination, validator.root):\n        raise ValidationError("Build output must be outside the submission directory")\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    with zipfile.ZipFile(destination, "w", compression=zipfile.ZIP_DEFLATED) as archive:\n        for name in manifest["package_files"]:\n            archive.write(validator.root / name, arcname=name)\n    return {**manifest, "archive": str(destination), "archive_bytes": destination.stat().st_size}\n\n\ndef _number(value: Any) -> float | int | None:\n    if isinstance(value, (float, int)) and not isinstance(value, bool) and math.isfinite(value):\n        return value\n    return None\n\n\ndef _pick(mapping: dict, *keys: str) -> Any:\n    for key in keys:\n        if mapping.get(key) is not None:\n            return mapping[key]\n    return None\n\n\ndef _truth(value: Any) -> bool | None:\n    if isinstance(value, bool):\n        return value\n    if value in (0, 1):\n        return bool(value)\n    if isinstance(value, str) and value.lower() in {"true", "false"}:\n        return value.lower() == "true"\n    return None\n\n\ndef _timestamp(value: Any) -> float | None:\n    if _number(value) is not None:\n        return float(value)\n    if isinstance(value, str):\n        try:\n            return datetime.fromisoformat(value.replace("Z", "+00:00")).timestamp()\n        except ValueError:\n            pass\n    return None\n\n\ndef _artifact(root: Path, subdir: str, filename: str) -> Path | None:\n    target = (root / subdir / filename).resolve()\n    return target if _inside(target, root) and target.is_file() and not target.is_symlink() else None\n\n\ndef _structured_result(value: Any) -> dict | None:\n    if isinstance(value, dict):\n        return value\n    if isinstance(value, str):\n        try:\n            result = json.loads(value)\n            return result if isinstance(result, dict) else None\n        except json.JSONDecodeError:\n            pass\n    return None\n\n\ndef trace_metrics(path: Path | None, analyzer_names: set[str], reviewer_names: set[str] | None = None,\n                  helper_names: set[str] | None = None) -> dict:\n    reviewer_names = reviewer_names or set()\n    helper_names = (helper_names or set()) | analyzer_names | reviewer_names\n    locator_names = {name for name in helper_names if "locator" in name.lower()}\n    verifier_names = {name for name in helper_names if "verifier" in name.lower()}\n    result = {\n        "trace_present": path is not None, "trace_valid": None,\n        "trace_schema_supported": None, "tool_calls_by_name": None,\n        "test_command_calls_observed": None,\n        "model_requests": None, "traced_tool_calls": None, "known_budgeted_tool_calls": None,\n        "analyzer_calls": None, "reviewer_calls": None, "tool_errors": None, "first_successful_edit_trace_seconds": None,\n        "locator_calls": None, "verifier_calls": None, "helper_calls_by_name": None,\n        "input_tokens_sum_observed": None, "output_tokens_sum_observed": None,\n        "max_request_input_tokens": None, "requests_with_input_tokens": None,\n        "trace_cumulative_token_metrics": None, "trace_warnings": [],\n    }\n    if path is None:\n        return result\n    try:\n        trace = json.loads(path.read_text(encoding="utf-8"))\n    except (OSError, UnicodeError, json.JSONDecodeError) as exc:\n        result["trace_valid"] = False\n        result["trace_warnings"].append(f"Unreadable trace: {type(exc).__name__}")\n        return result\n    if not isinstance(trace, dict) or not isinstance(trace.get("steps"), list):\n        result["trace_valid"] = False\n        result["trace_warnings"].append("Unsupported trace shape; expected a mapping with steps[]")\n        return result\n    result["trace_valid"] = True\n    cumulative = trace.get("final_metrics", trace.get("metrics"))\n    if isinstance(cumulative, dict):\n        result["trace_cumulative_token_metrics"] = {\n            key: value for key, value in cumulative.items() if "token" in key.lower() and _number(value) is not None\n        } or None\n    calls: list[dict] = []\n    observations: dict[str, dict] = {}\n    inputs: list[int | float] = []\n    outputs: list[int | float] = []\n    requests = 0\n    malformed_calls = 0\n    recognized = not trace["steps"]\n    canonical = all(isinstance(step, dict) and step.get("source", step.get("role")) in ("agent", "assistant", "tool", "user", "system") for step in trace["steps"])\n    trace_start = _timestamp(_pick(trace, "start_time", "timestamp", "task_start_time"))\n    for step in trace["steps"]:\n        if not isinstance(step, dict):\n            result["trace_warnings"].append("Non-mapping step ignored")\n            continue\n        source = step.get("source", step.get("role"))\n        kind = step.get("type", step.get("step_type"))\n        model_step = source in ("agent", "assistant") or kind in ("model_response", "llm_response", "model_call", "llm_call")\n        if model_step:\n            recognized = True\n            requests += 1\n            usage = next((step[key] for key in ("metrics", "usage", "token_usage", "usage_metadata") if isinstance(step.get(key), dict)), {})\n            input_tokens = _number(_pick(usage, "prompt_tokens", "input_tokens", "prompt_token_count"))\n            output_tokens = _number(_pick(usage, "completion_tokens", "output_tokens", "candidates_token_count"))\n            if input_tokens is not None:\n                inputs.append(input_tokens)\n            if output_tokens is not None:\n                outputs.append(output_tokens)\n        elapsed = _number(_pick(step, "elapsed_seconds", "relative_seconds"))\n        if elapsed is None and trace_start is not None:\n            stamp = _timestamp(step.get("timestamp"))\n            elapsed = stamp - trace_start if stamp is not None and stamp >= trace_start else None\n        declared = step.get("tool_calls")\n        if isinstance(declared, list):\n            recognized = True\n        elif isinstance(step.get("tool_call"), dict):\n            declared = [step["tool_call"]]\n            recognized = True\n        elif kind == "tool_call" and isinstance(step.get("tool_name"), str):\n            declared = [step]\n            recognized = True\n        else:\n            declared = []\n        for call in declared:\n            if not isinstance(call, dict):\n                result["trace_warnings"].append("Non-mapping tool call ignored")\n                malformed_calls += 1\n                continue\n            function = call.get("function") if isinstance(call.get("function"), dict) else {}\n            name = _pick(call, "function_name", "tool_name", "name") or function.get("name")\n            if not isinstance(name, str):\n                result["trace_warnings"].append("Tool call with unknown name ignored")\n                malformed_calls += 1\n                continue\n            arguments = _structured_result(call.get("arguments", function.get("arguments"))) or {}\n            calls.append({"name": name, "id": str(_pick(call, "tool_call_id", "call_id", "id") or ""),\n                          "elapsed": elapsed, "result": _structured_result(_pick(call, "result", "output")),\n                          "command": arguments.get("command")})\n        observation = step.get("observation")\n        if isinstance(observation, dict) and isinstance(observation.get("results"), list):\n            for observed in observation["results"]:\n                if isinstance(observed, dict):\n                    identifier = _pick(observed, "source_call_id", "tool_call_id", "call_id")\n                    value = _structured_result(_pick(observed, "content", "result", "output"))\n                    if identifier is not None and value is not None:\n                        observations[str(identifier)] = value\n        if source == "tool" or kind in ("tool_result", "tool_response"):\n            identifier = _pick(step, "tool_call_id", "source_call_id", "call_id")\n            value = _structured_result(_pick(step, "content", "message", "result", "output"))\n            if identifier is not None and value is not None:\n                observations[str(identifier)] = value\n    if not recognized:\n        result["trace_schema_supported"] = False\n        result["trace_warnings"].append("Unrecognized step schema; request/tool metrics unavailable")\n        return result\n    result["trace_schema_supported"] = True\n    frequency = dict(sorted(Counter(call["name"] for call in calls).items()))\n    unknown_names = set(frequency) - CORE_TOOLS - SKILL_TOOLS - helper_names\n    if unknown_names:\n        result["trace_warnings"].append("Unclassified tool names (may be other AgentTools): " + ", ".join(sorted(unknown_names)))\n    if malformed_calls:\n        result["trace_warnings"].append("Tool-call counts are partial because some calls could not be parsed")\n    result.update({\n        "model_requests": requests if requests or canonical else None, "traced_tool_calls": len(calls),\n        "tool_calls_by_name": frequency,\n        "test_command_calls_observed": sum(call["name"] == "run_command" and isinstance(call["command"], str) and bool(re.search(r"\\b(?:pytest|unittest|nosetests|tox)\\b", call["command"])) for call in calls),\n        "known_budgeted_tool_calls": sum(c["name"] in ((CORE_TOOLS - FREE_TOOLS) | SKILL_TOOLS) for c in calls),\n        "analyzer_calls": sum(c["name"] in analyzer_names for c in calls),\n        "reviewer_calls": sum(c["name"] in reviewer_names for c in calls),\n        "locator_calls": sum(c["name"] in locator_names for c in calls),\n        "verifier_calls": sum(c["name"] in verifier_names for c in calls),\n        "helper_calls_by_name": {name: frequency.get(name, 0) for name in sorted(helper_names)},\n        "requests_with_input_tokens": len(inputs),\n        "input_tokens_sum_observed": sum(inputs) if inputs else None,\n        "output_tokens_sum_observed": sum(outputs) if outputs else None,\n        "max_request_input_tokens": max(inputs) if inputs else None,\n    })\n    statuses = []\n    edit_times = []\n    for call in calls:\n        observed = call["result"] or observations.get(call["id"])\n        if observed is None:\n            continue\n        status = observed.get("status")\n        if status in ("ok", "success", "error"):\n            statuses.append(status)\n        if call["name"] in {"edit_file", "write_file"} and status in ("ok", "success") and call["elapsed"] is not None:\n            edit_times.append(call["elapsed"])\n    result["tool_errors"] = sum(status == "error" for status in statuses) if statuses else None\n    result["first_successful_edit_trace_seconds"] = min(edit_times) if edit_times else None\n    if requests and len(inputs) != requests:\n        result["trace_warnings"].append("Per-request input token metrics are incomplete")\n    if calls and len(statuses) != len(calls):\n        result["trace_warnings"].append("Some tool outcomes are unavailable; successful edit time may be incomplete")\n    return result\n\n\nSTOP_PATTERNS = [\n    ("time", r"TimeExceeded|AgentTimedOut|agent (?:session )?timed? ?out|(?:agent|session) timeout|time (?:budget|limit|allowance).{0,35}(?:exhaust|exceed|reached)|(?:exhaust|exceed).{0,25}time budget"),\n    ("tool_budget", r"(?:tool.?calls? (?:budget|limit)|max_tool_calls).{0,35}(?:exhaust|exceed|reached)|(?:exhaust|exceed).{0,25}tool.?call"),\n    ("turn_budget", r"(?:turn (?:budget|limit)|max_turns|max_loop_iterations).{0,35}(?:exhaust|exceed|reached)"),\n    ("context", r"ContextWindowExceeded|context (?:window|length).{0,30}(?:exceed|overflow)|maximum context length"),\n    ("setup", r"(?:failed|unable) to (?:create|start|prepare).{0,20}(?:sandbox|container)|(?:snapshot|wheel|dependency|container) (?:setup|installation|extraction) failed|ContainerSetupError|SandboxSetupError"),\n]\n\n\ndef classify(record: dict, patch_present: bool | None, agent_log: str, test_log: str) -> tuple[str, str, list[str]]:\n    error = record.get("error")\n    metadata = " ".join(str(record.get(k) or "") for k in ("stop_reason", "exit_status", "error_type", "error", "stage"))\n    stop = "unknown"\n    evidence: list[str] = []\n    explicit_stop = _pick(record, "stop_reason", "exit_status")\n    if isinstance(explicit_stop, str) and explicit_stop.lower() in {"submitted", "complete", "completed", "finished"}:\n        stop = "submitted"\n        evidence.append(f"Recorded stop: {explicit_stop}")\n    # Metadata is authoritative. Transcript fallback requires a harness logging\n    # severity prefix, rather than a matching phrase in model thoughts/tool output.\n    qualified_log = "\\n".join(line for line in agent_log.splitlines() if re.search(r"(?:^|\\s|\\[)(?:ERROR|WARNING|CRITICAL)(?:\\s|:|\\])", line))\n    for name, pattern in STOP_PATTERNS:\n        match = re.search(pattern, metadata, re.I)\n        if match is None and stop == "unknown":\n            match = re.search(pattern, qualified_log, re.I)\n        if match:\n            stop = name\n            evidence.append(match.group(0)[:200])\n            break\n    if stop == "unknown" and error:\n        if re.search(r"FileEditError|ToolExecutionError|invalid tool call|malformed tool|unknown tool|tool.call.{0,20}truncat", metadata, re.I):\n            stop = "tool_error"\n        elif re.search(r"timeout|timed out", metadata, re.I):\n            stop = "timeout_unspecified"\n        evidence.append(str(error)[:400])\n    resolved = _truth(record.get("resolved"))\n    if resolved is True:\n        return "solved", stop, evidence\n    if re.search(r"failed to apply (?:agent )?patch|PatchApplyError", metadata + "\\n" + test_log, re.I):\n        return "patch_application_failure", stop, evidence + ["Patch application error recorded"]\n    if stop in {"setup", "context", "tool_budget", "turn_budget", "tool_error", "timeout_unspecified"}:\n        return stop, stop, evidence\n    if stop == "time":\n        return ("timeout_with_patch" if patch_present is True else "timeout_before_patch" if patch_present is False else "timeout_patch_unknown"), stop, evidence\n    if patch_present is False:\n        return "empty_patch", stop, evidence + ["Recorded patch is empty"]\n    exit_code = _number(_pick(record, "test_exit_code", "verification_exit_code"))\n    if resolved is False and exit_code is not None and exit_code >= 0:\n        if re.search(r"ImportError|ModuleNotFoundError|collected 0 items", test_log):\n            return "verification_environment_failure", stop, evidence + ["Verification output reports import/collection failure"]\n        return "verification_failure", stop, evidence + [f"Unresolved with verification exit code {exit_code}"]\n    if error:\n        return "reported_error", stop, evidence\n    return "unknown", stop, evidence\n\n\ndef verification_status(record: dict, patch_present: bool | None) -> str:\n    if _truth(record.get("resolved")) is True:\n        return "passed"\n    error = str(record.get("error") or "")\n    if re.search(r"failed to apply (?:agent )?patch|PatchApplyError", error, re.I):\n        return "not_run"\n    code = _number(_pick(record, "test_exit_code", "verification_exit_code"))\n    if code is not None and code >= 0:\n        return "failed" if code != 0 or _truth(record.get("resolved")) is False else "unknown"\n    if patch_present is False or code == -1:\n        return "not_run"\n    return "unknown"\n\n\nSUGGESTIONS = {\n    "solved": "Retain as regression coverage; compare runtime and calls with the control.",\n    "timeout_before_patch": "Inspect localization and model latency; compare a larger agent allowance with the same task.",\n    "timeout_with_patch": "Check verification and completion discipline before changing adapters or architecture.",\n    "timeout_patch_unknown": "Recover the patch artifact and session trace before diagnosing the timeout.",\n    "timeout_unspecified": "Determine whether the timeout affected a command, inference request, setup, or whole session.",\n    "tool_budget": "Inspect repeated navigation and unsuccessful edits; reserve calls for verification.",\n    "turn_budget": "Inspect repeated responses and incomplete tool calls before increasing turn limits.",\n    "context": "Reduce broad reads and preserve a concise task state; compare selective analyzer delegation.",\n    "setup": "Check snapshots, wheels, sandbox backend, and environment; this is not a patch-quality result.",\n    "tool_error": "Inspect tool arguments and call truncation; use smaller uniquely targeted edits.",\n    "empty_patch": "Check premature completion, repeated searches, and whether any edit succeeded.",\n    "patch_application_failure": "Inspect patch paths and formatting against the task baseline.",\n    "verification_failure": "Inspect targeted test output and task requirements; exit zero alone is insufficient.",\n    "verification_environment_failure": "Separate dependency/collection problems from incorrect task logic.",\n    "reported_error": "Read the recorded error and task artifacts; the failure is not classified further.",\n    "unknown": "Capture the final stop reason, agent elapsed time, trace, patch, and verification output.",\n}\n\n\ndef make_report(results_dir: str | Path, output_dir: str | Path, submission_dir: str | Path | None = None) -> dict:\n    root = Path(results_dir).resolve()\n    source = root / "notebook_task_results.jsonl"\n    if not source.is_file():\n        source = root / "task_results.jsonl"\n    if not source.is_file():\n        raise ValueError(f"Missing task_results.jsonl: {root}")\n    warnings: list[str] = []\n    records: dict[str, dict] = {}\n    for line_number, line in enumerate(source.read_text(encoding="utf-8").splitlines(), 1):\n        if not line.strip():\n            continue\n        try:\n            row = json.loads(line)\n            if not isinstance(row, dict):\n                raise ValueError("row is not a mapping")\n            identifier = _pick(row, "instance_id", "task_id", "id")\n            if not isinstance(identifier, (str, int)) or not str(identifier):\n                raise ValueError("row has no task identifier")\n            identifier = str(identifier)\n            if identifier in records:\n                warnings.append(f"Duplicate task {identifier}: report uses the last finished record")\n            records[identifier] = row\n        except (json.JSONDecodeError, ValueError) as exc:\n            warnings.append(f"task_results.jsonl line {line_number}: {exc}")\n    configuration = validate_submission(submission_dir) if submission_dir else None\n    analyzer_names = {"code_analyzer_agent"}\n    reviewer_names = {"patch_reviewer_agent"}\n    helper_names = {"repository_locator_agent", "test_verifier_agent"}\n    if configuration:\n        analyzer_names |= {name for name in configuration["agent_tool_names"] if "analy" in name.lower()}\n        reviewer_names |= {name for name in configuration["agent_tool_names"] if "review" in name.lower()}\n        helper_names |= set(configuration["agent_tool_names"])\n    tasks = []\n    for identifier, original in records.items():\n        row = dict(original)\n        raw = row.get("raw_result", row.get("metadata"))\n        if isinstance(raw, dict):\n            for key in ("error", "error_type", "exit_status", "stop_reason", "stage", "agent_elapsed_seconds"):\n                if row.get(key) is None and raw.get(key) is not None:\n                    row[key] = raw[key]\n        patch_path = _artifact(root, "patches", f"{identifier}.patch")\n        agent_paths = [path for filename in (f"{identifier}.log", f"observer_{identifier}.log", f"exception_{identifier}.log") if (path := _artifact(root, "logs", filename)) is not None]\n        test_path = _artifact(root, "test_outputs", f"{identifier}.log")\n        trace_path = _artifact(root, "traces", f"trace_{identifier}.json")\n        patch = row.get("agent_patch", row.get("prediction"))\n        sentinel = isinstance(patch, str) and patch.strip() == "NO_PATCH"\n        patch_chars = (0 if sentinel else len(patch)) if isinstance(patch, str) else _number(_pick(row, "patch_chars", "patch_size"))\n        patch_bytes = patch_path.stat().st_size if patch_path else None\n        file_sentinel = patch_path is not None and patch_bytes is not None and patch_bytes < 100 and patch_path.read_text(encoding="utf-8", errors="replace").strip() == "NO_PATCH"\n        file_patch_present = False if file_sentinel else patch_bytes > 0 if patch_bytes is not None else None\n        patch_present = patch_chars > 0 if patch_chars is not None else file_patch_present\n        if patch_chars is not None and file_patch_present is not None and patch_present != file_patch_present:\n            warnings.append(f"{identifier}: record and patch artifact disagree on patch presence; record used")\n        agent_log = "\\n".join(path.read_text(encoding="utf-8", errors="replace") for path in agent_paths)\n        test_log = test_path.read_text(encoding="utf-8", errors="replace") if test_path else ""\n        category, stop, evidence = classify(row, patch_present, agent_log, test_log)\n        observed = trace_metrics(trace_path, analyzer_names, reviewer_names, helper_names)\n        tasks.append({\n            "instance_id": identifier, "repo": row.get("repo"), "resolved": _truth(row.get("resolved")),\n            "test_exit_code": _number(_pick(row, "test_exit_code", "verification_exit_code")),\n            "tool_calls": _number(_pick(row, "tool_calls", "tool_calls_used")),\n            "duration_seconds": _number(_pick(row, "duration_seconds", "duration")),\n            "agent_elapsed_seconds": _number(row.get("agent_elapsed_seconds")),\n            "observer_wall_seconds": _number(row.get("observer_wall_seconds")),\n            "patch_present": patch_present, "patch_chars": patch_chars, "patch_bytes": patch_bytes,\n            "failure_category": category, "stop_cause": stop, "evidence": evidence,\n            "verification_status": verification_status(row, patch_present),\n            "suggestion": SUGGESTIONS[category],\n            "error": str(row["error"])[:600] if row.get("error") else None,\n            "agent_log_present": bool(agent_paths), "test_output_present": test_path is not None,\n            **observed,\n        })\n    count = len(tasks)\n    known = sum(task["resolved"] is not None for task in tasks)\n    solved = sum(task["resolved"] is True for task in tasks)\n    per_repository = {}\n    for repo in sorted({str(task["repo"]) if task["repo"] is not None else "unknown" for task in tasks}):\n        subset = [task for task in tasks if (str(task["repo"]) if task["repo"] is not None else "unknown") == repo]\n        known_count = sum(task["resolved"] is not None for task in subset)\n        solved_count = sum(task["resolved"] is True for task in subset)\n        per_repository[repo] = {\n            "count": len(subset), "known_resolution_count": known_count, "resolved": solved_count,\n            "resolution_rate": solved_count / len(subset) if known_count == len(subset) else None,\n            "resolution_rate_known_records": solved_count / known_count if known_count else None,\n        }\n    summary = {\n        "finished_task_records": count, "known_resolution_records": known,\n        "resolved": solved, "unresolved": sum(task["resolved"] is False for task in tasks),\n        "resolution_rate": solved / count if count and known == count else None,\n        "resolution_rate_known_records": solved / known if known else None,\n        "trace_present": sum(task["trace_present"] for task in tasks),\n        "trace_valid": sum(task["trace_valid"] is True for task in tasks),\n        "trace_schema_supported": sum(task["trace_schema_supported"] is True for task in tasks),\n        "nonempty_patch_records": sum(task["patch_present"] is True for task in tasks),\n        "patch_presence_unknown": sum(task["patch_present"] is None for task in tasks),\n        "failure_categories": dict(sorted(Counter(task["failure_category"] for task in tasks).items())),\n        "per_repository": per_repository,\n        "tool_calls_by_name_observed": dict(sorted(sum((Counter(task["tool_calls_by_name"] or {}) for task in tasks), Counter()).items())),\n    }\n    report = {\n        "results_dir": str(root), "source": str(source), "summary": summary,\n        "configuration": configuration, "tasks": tasks, "warnings": warnings,\n        "interpretation": [\n            "Rates cover finished records only; missing/unstarted tasks are not in this denominator.",\n            "duration_seconds is the recorded total task duration, not inferred agent-loop time.",\n            "Missing metrics remain null; unsupported or incomplete traces cannot establish zero usage.",\n            "max_request_input_tokens uses per-agent-step input usage only; cumulative tokens are separate.",\n            "first_successful_edit_trace_seconds is relative to the trace clock, not necessarily the agent-session clock.",\n            "known_budgeted_tool_calls counts recognized chargeable built-ins/skill tools; get_status and submit_patch are free.",\n            "Analyzer calls are identified by known configured analyzer names; delegation does not imply a separate model.",\n            "Reviewer calls are identified by known configured reviewer names; a call count does not establish review benefit or successful repair.",\n            "Locator/verifier calls and helper_calls_by_name count observed configured delegation names; missing trace usage remains unknown.",\n            "Tool frequencies cover parsed trace calls; test_command_calls_observed recognizes named pytest/unittest/nosetests/tox commands only.",\n            "Stop causes require explicit recorded error/exhaustion evidence, not duration or budget comparisons.",\n        ],\n    }\n    destination = Path(output_dir).resolve()\n    destination.mkdir(parents=True, exist_ok=True)\n    (destination / "diagnostics.json").write_text(json.dumps(report, ensure_ascii=False, indent=2) + "\\n", encoding="utf-8")\n    fields = [\n        "instance_id", "repo", "resolved", "failure_category", "stop_cause", "verification_status", "test_exit_code",\n        "patch_present", "patch_chars", "patch_bytes", "duration_seconds", "agent_elapsed_seconds", "observer_wall_seconds",\n        "tool_calls", "model_requests", "traced_tool_calls", "known_budgeted_tool_calls", "analyzer_calls", "reviewer_calls",\n        "locator_calls", "verifier_calls",\n        "first_successful_edit_trace_seconds", "max_request_input_tokens", "input_tokens_sum_observed",\n        "output_tokens_sum_observed", "requests_with_input_tokens", "trace_present", "trace_valid",\n        "trace_schema_supported", "test_command_calls_observed",\n    ]\n    with (destination / "diagnostics.csv").open("w", encoding="utf-8", newline="") as handle:\n        writer = csv.DictWriter(handle, fieldnames=fields, extrasaction="ignore")\n        writer.writeheader()\n        writer.writerows(tasks)\n    rate = f"{summary[\'resolution_rate\']:.1%}" if summary["resolution_rate"] is not None else "unknown"\n    markdown = [\n        "# Evaluation diagnostics", "", f"Resolved **{solved}/{count}** finished records; recorded resolution rate: **{rate}**.",\n        f"Traces present: {summary[\'trace_present\']}/{count}; valid envelopes: {summary[\'trace_valid\']}/{count}; recognized step schemas: {summary[\'trace_schema_supported\']}/{count}.", "",\n        "| Task | Resolved | Failure category | Verification | Stop evidence | Total task seconds | Agent seconds |", "|---|---|---|---|---|---:|---:|",\n    ]\n    for task in tasks:\n        evidence = "; ".join(task["evidence"]) or "Not recorded"\n        values = [task["instance_id"], task["resolved"], task["failure_category"], task["verification_status"], evidence, task["duration_seconds"], task["agent_elapsed_seconds"]]\n        markdown.append("| " + " | ".join("unknown" if value is None else str(value).replace("|", "\\\\|").replace("\\n", " ").replace("\\r", " ") for value in values) + " |")\n    markdown.extend(["", "## Suggested checks", ""])\n    for category in summary["failure_categories"]:\n        markdown.append(f"- **{category}**: {SUGGESTIONS[category]}")\n    markdown.extend(["", "## Observed tool calls", ""])\n    frequency = summary["tool_calls_by_name_observed"]\n    markdown.extend([f"- `{name}`: {frequency[name]}" for name in frequency] or ["No parsed tool-call records available."])\n    markdown.extend(["", "## Repository results", "", "| Repository | Resolved | Known outcomes | Finished records |", "|---|---:|---:|---:|"])\n    for repo, stats in per_repository.items():\n        markdown.append(f"| {repo.replace(\'|\', \'/\')} | {stats[\'resolved\']} | {stats[\'known_resolution_count\']} | {stats[\'count\']} |")\n    trace_warnings = [f"{task[\'instance_id\']}: {warning}" for task in tasks for warning in task["trace_warnings"]]\n    if trace_warnings:\n        markdown.extend(["", "## Trace warnings", "", *[f"- {warning}" for warning in trace_warnings]])\n    markdown.extend(["", "## Limits of these observations", "", *[f"- {item}" for item in report["interpretation"]]])\n    if warnings:\n        markdown.extend(["", "## Input warnings", "", *[f"- {item}" for item in warnings]])\n    (destination / "diagnostics.md").write_text("\\n".join(markdown) + "\\n", encoding="utf-8")\n    return report\n\n\ndef main(argv: list[str] | None = None) -> int:\n    parser = argparse.ArgumentParser(description=__doc__)\n    subparsers = parser.add_subparsers(dest="command", required=True)\n    for name in ("validate", "build"):\n        child = subparsers.add_parser(name)\n        child.add_argument("--submission-dir", required=True)\n        if name == "build":\n            child.add_argument("--output", required=True)\n    report = subparsers.add_parser("report")\n    report.add_argument("--results-dir", required=True)\n    report.add_argument("--output-dir", required=True)\n    report.add_argument("--submission-dir")\n    args = parser.parse_args(argv)\n    try:\n        if args.command == "validate":\n            result = validate_submission(args.submission_dir)\n        elif args.command == "build":\n            result = build_submission(args.submission_dir, args.output)\n        else:\n            result = make_report(args.results_dir, args.output_dir, args.submission_dir)["summary"]\n        print(json.dumps(result, ensure_ascii=False, indent=2))\n        return 0\n    except (ValidationError, ValueError, OSError, yaml.YAMLError) as exc:\n        print(json.dumps({"valid": False, "error": str(exc)}, ensure_ascii=False), file=sys.stderr)\n        return 1\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n'
EMBEDDED_SOURCE_HASHES = {'agent.yaml': 'f051358efae782f8b6f89252d15dbd3ae6d2876aa01df75332dbecce7536b55a', 'configs/sampling.yaml': '6c2d8ef489b6c15c6a57e18b01eb46e35a704b8142f93914f08b0e68d0d352cf', 'eval_config.yaml': 'b2097fb8f21c29a10cae1ee212ed39ab60cec0c2660e37dd13c5a13ce0aafe89', 'prompts/system.md': 'cc8c45b677f9e7d44e63ffd1ba5f7732ff0dae73c4a7cfa3ff7a7b0ebbd7748b', 'sub_agents/locator_resources/instruction.md': 'e1bbb6e72a79a07801a3c8828282e9786ee28cf393d959bb6556dafdc629f196', 'sub_agents/locator_resources/sampling.yaml': '636ab5d342e3057c9a4fa43917ff66401fe55b4d7976f3f7c31e52da021e3688', 'sub_agents/repository_locator.yaml': 'd101acb0bc9809bc9d398c71bfff1cf90fc57421683441ed61eca8ccdb2b09c5', 'sub_agents/test_verifier.yaml': '666ba8189d05dfde61eb5d3b32c651df5ba46ad494baf869855d11a88a2c3799', 'sub_agents/verifier_resources/instruction.md': '2ec6abd1f20dd1391cb18c0ce3bc62ba53bf3a4f64febae3c9535049ccd937c8', 'sub_agents/verifier_resources/sampling.yaml': '636ab5d342e3057c9a4fa43917ff66401fe55b4d7976f3f7c31e52da021e3688', 'notebook_tools/submission_tools.py': '8a9e27629e719474cfb922b2c5c577e943a331e548f935a274976b9fe6b53159'}
EMBEDDED_BASELINE_HASHES = {'agent.yaml': 'f051358efae782f8b6f89252d15dbd3ae6d2876aa01df75332dbecce7536b55a', 'configs/sampling.yaml': '3dab0b2506dae34ce92fef7b380d6073729104b23fe2cf66d44ac1d2f213aa6a', 'eval_config.yaml': '9917f55b924efb5588b6ce870af7364558f6798d91590ef1579aec643f5245a4', 'prompts/system.md': 'cc8c45b677f9e7d44e63ffd1ba5f7732ff0dae73c4a7cfa3ff7a7b0ebbd7748b', 'sub_agents/locator_resources/instruction.md': 'e1bbb6e72a79a07801a3c8828282e9786ee28cf393d959bb6556dafdc629f196', 'sub_agents/locator_resources/sampling.yaml': 'c4a99cfb61cfeeae03200c49d61d2f66957d1d5b663e2f0a448fb83227ca5bba', 'sub_agents/repository_locator.yaml': 'd101acb0bc9809bc9d398c71bfff1cf90fc57421683441ed61eca8ccdb2b09c5', 'sub_agents/test_verifier.yaml': '666ba8189d05dfde61eb5d3b32c651df5ba46ad494baf869855d11a88a2c3799', 'sub_agents/verifier_resources/instruction.md': '2ec6abd1f20dd1391cb18c0ce3bc62ba53bf3a4f64febae3c9535049ccd937c8', 'sub_agents/verifier_resources/sampling.yaml': 'c4a99cfb61cfeeae03200c49d61d2f66957d1d5b663e2f0a448fb83227ca5bba'}
COMPARISON_MODE = 'runtime_comparison'
CANDIDATE_LABEL = 'submission_5'
BASELINE_LABEL = 'submission_4'


In [ ]:
def remove_agent_tools(text, config_paths=None):
    """Remove all or selected AgentTools while preserving !include YAML nodes."""
    node = yaml.compose(text)
    if not isinstance(node, yaml.MappingNode):
        raise ValueError("agent.yaml must be a mapping")
    removed = 0
    for key, value in node.value:
        if key.value == "tools":
            if not isinstance(value, yaml.SequenceNode):
                raise ValueError("tools must be a sequence")
            keep = []
            for item in value.value:
                is_delegate = isinstance(item, yaml.MappingNode) and any(
                    k.value == "agent_tool" for k, _ in item.value
                )
                should_remove = is_delegate
                if is_delegate and config_paths is not None:
                    spec = next(v for k, v in item.value if k.value == "agent_tool")
                    path = next((v.value for k, v in spec.value if k.value == "config_path"), None)
                    should_remove = path in config_paths
                removed += int(should_remove)
                if not should_remove:
                    keep.append(item)
            value.value = keep
    if not removed:
        raise ValueError("Candidate has no matching agent_tool to remove")
    return yaml.serialize(node)


def task_resources(task, data_dir):
    """Check task-named/commit-named files described by the public dataset."""
    def field(name):
        return task[name] if isinstance(task, dict) else getattr(task, name)

    identifier = field("instance_id")
    repo_short = field("repo").split("/")[-1]
    commit_stem = repo_short + "_" + field("base_commit")
    root = Path(data_dir)
    choices = {
        "snapshot": [root / "snapshots" / (identifier + ".tgz")],
        "graph": [root / "graphs" / (commit_stem + ".json"), root / "graphs" / (identifier + ".json")],
        "embedding": [root / "embeddings" / (commit_stem + ".npz"), root / "embeddings" / (identifier + ".npz")],
    }
    return {kind: next((str(p) for p in paths if p.is_file() and p.stat().st_size > (0 if kind == 'snapshot' else 100)), None)
            for kind, paths in choices.items()}


def select_tasks(tasks, data_dir, task_ids=None, limit=2):
    """Select complete tasks deterministically, round robin by repository."""
    def field(task, name):
        return task[name] if isinstance(task, dict) else getattr(task, name)

    if limit is not None and (not isinstance(limit, int) or isinstance(limit, bool) or limit <= 0):
        raise ValueError("TASK_LIMIT must be a positive integer or None")
    indexed = {field(t, "instance_id"): t for t in tasks}
    if len(indexed) != len(tasks):
        raise ValueError("Duplicate instance_id in tasks.jsonl")
    coverage = {identifier: task_resources(t, data_dir) for identifier, t in indexed.items()}
    complete = {identifier for identifier, resources in coverage.items() if all(resources.values())}
    if task_ids:
        if len(set(task_ids)) != len(task_ids):
            raise ValueError("TASK_IDS contains duplicates")
        missing = {identifier: coverage.get(identifier, "unknown task") for identifier in task_ids
                   if identifier not in complete}
        if missing:
            raise ValueError("TASK_IDS lack complete matching resources: " + json.dumps(missing))
        selected = [indexed[identifier] for identifier in task_ids]
    else:
        by_repo = defaultdict(list)
        for identifier in sorted(complete):
            task = indexed[identifier]
            by_repo[field(task, "repo")].append(task)
        selected = []
        while any(by_repo.values()):
            for repo in sorted(by_repo):
                if by_repo[repo]:
                    selected.append(by_repo[repo].pop(0))
        if limit is not None:
            selected = selected[:limit]
    if not selected:
        raise ValueError("No task has matching snapshot, graph and embedding files. Attach the full competition dataset.")
    return selected, coverage


def extract_curated(archive_path, destination):
    """Extract a curated archive to a fresh directory, validating member paths."""
    import shutil

    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=False)
    resolved = destination.resolve()
    with zipfile.ZipFile(archive_path) as archive:
        for info in archive.infolist():
            target = (resolved / info.filename).resolve()
            if not target.is_relative_to(resolved) or "\\" in info.filename:
                raise ValueError("Unsafe archive member: " + info.filename)
            if info.is_dir():
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(info) as source, target.open('wb') as output:
                    shutil.copyfileobj(source, output, length=1024 * 1024)


def file_sha256(path):
    """Hash large adapter assets without materializing the whole file."""
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


In [ ]:
from datetime import datetime, timezone
import hashlib
import importlib.util
import json
import logging
import shutil
import subprocess
import sys
import tempfile
import zipfile
from collections import defaultdict
import yaml

if SANDBOX not in {'subprocess', 'docker'}:
    raise ValueError('SANDBOX must be subprocess or docker')
if not BENCHMARK_VARIANTS or len(set(BENCHMARK_VARIANTS)) != len(BENCHMARK_VARIANTS):
    raise ValueError('Choose unique benchmark variants')
allowed_variants = {
    'prompt_comparison': {'baseline', 'candidate'},
    'runtime_comparison': {'baseline', 'timeout_only', 'thinking_off_only', 'candidate'},
    'reviewer_control': {'coder_only', 'optional_analyzer', 'optional_reviewer'},
    'verification_control': {'coder_only', 'optional_locator', 'optional_verifier', 'locator_verifier'},
    'architecture_control': {'coder_only', 'optional_analyzer'},
}[COMPARISON_MODE]
if set(BENCHMARK_VARIANTS) - allowed_variants:
    raise ValueError('Unknown benchmark variant')
if RELEASE_VARIANT not in BENCHMARK_VARIANTS:
    raise ValueError('RELEASE_VARIANT must be evaluated in this run')
run_parent = WORKING_DIR / 'agent_runs'
run_parent.mkdir(parents=True, exist_ok=True)
run_prefix = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S.%fZ_')
RUN_DIR = Path(tempfile.mkdtemp(prefix=run_prefix, dir=run_parent))
AUX_DIR = RUN_DIR / 'tools'
AUX_DIR.mkdir()
TOOL_SCRIPT = AUX_DIR / 'submission_tools.py'
TOOL_SCRIPT.write_text(EMBEDDED_TOOL_SOURCE, encoding='utf-8')

def tool_cli(*args):
    process = subprocess.run([sys.executable, str(TOOL_SCRIPT), *map(str, args)],
                             capture_output=True, text=True)
    if process.returncode:
        raise RuntimeError(process.stderr or process.stdout)
    return json.loads(process.stdout)

candidate_variant = {
    'prompt_comparison': 'candidate', 'reviewer_control': 'optional_reviewer',
    'runtime_comparison': 'candidate',
    'verification_control': 'locator_verifier', 'architecture_control': 'optional_analyzer',
}[COMPARISON_MODE]
candidate_dir = RUN_DIR / 'variants' / candidate_variant
candidate_zip = RUN_DIR / 'candidate_source.zip'
if SUBMISSION_SOURCE_DIR is not None:
    source = Path(SUBMISSION_SOURCE_DIR)
    if not (source / 'agent.yaml').is_file():
        raise ValueError('SUBMISSION_SOURCE_DIR must contain your candidate agent.yaml')
    tool_cli('build', '--submission-dir', source, '--output', candidate_zip)
    extract_curated(candidate_zip, candidate_dir)
else:
    candidate_dir.mkdir(parents=True, exist_ok=False)
    for name, content in EMBEDDED_SUBMISSION.items():
        target = candidate_dir / name
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(content, encoding='utf-8')

VARIANT_DIRS = {candidate_variant: candidate_dir}
if COMPARISON_MODE in {'prompt_comparison', 'runtime_comparison'}:
    baseline_dir = RUN_DIR / 'variants' / 'baseline'
    baseline_dir.mkdir(parents=True, exist_ok=False)
    for name, content in EMBEDDED_BASELINE.items():
        target = baseline_dir / name
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(content, encoding='utf-8')
    VARIANT_DIRS['baseline'] = baseline_dir
controls = {} if COMPARISON_MODE in {'prompt_comparison', 'runtime_comparison'} else {'coder_only': None}
if COMPARISON_MODE == 'runtime_comparison':
    # Restrict this experiment to two runtime changes, preserving all agent code/text.
    source_manifests = {name: tool_cli('validate', '--submission-dir', VARIANT_DIRS[name])
                        for name in ['baseline', 'candidate']}
    if any(manifest['agents'] != 3 or manifest['adapters'] for manifest in source_manifests.values()):
        raise ValueError('Runtime comparison requires unchanged-model three-agent submissions')
    package_files = set(source_manifests['baseline']['package_files'])
    if package_files != set(source_manifests['candidate']['package_files']):
        raise ValueError('Runtime comparison requires matching package paths')
    sampling_files = {name for name in package_files if name.endswith('sampling.yaml')}
    if len(sampling_files) != 3:
        raise ValueError('Runtime comparison requires three sampling files')
    for relative in sorted(package_files):
        baseline_text = (baseline_dir / relative).read_text(encoding='utf-8')
        candidate_text = (candidate_dir / relative).read_text(encoding='utf-8')
        if relative == 'eval_config.yaml':
            baseline_config, candidate_config = yaml.safe_load(baseline_text), yaml.safe_load(candidate_text)
            baseline_timeout = baseline_config['evaluation'].pop('timeout_seconds', None)
            candidate_timeout = candidate_config['evaluation'].pop('timeout_seconds', None)
            if baseline_timeout != 60 or candidate_timeout != 300 or baseline_config != candidate_config:
                raise ValueError('Runtime comparison permits only command/grading timeout 60 to 300')
        elif relative in sampling_files:
            baseline_config, candidate_config = yaml.safe_load(baseline_text), yaml.safe_load(candidate_text)
            baseline_thinking = baseline_config.pop('thinking_config', {})
            candidate_thinking = candidate_config.pop('thinking_config', {})
            if baseline_thinking.get('include_thoughts') is not True or candidate_thinking != {'include_thoughts': False} or baseline_config != candidate_config:
                raise ValueError('Runtime comparison permits only disabling thinking with other sampling fixed')
        elif baseline_text != candidate_text:
            raise ValueError('Runtime comparison requires identical agent configs, tools and prompts: ' + relative)
    for name, copied_files in [('timeout_only', {'eval_config.yaml'}), ('thinking_off_only', sampling_files)]:
        directory = RUN_DIR / 'variants' / name
        shutil.copytree(baseline_dir, directory)
        for relative in copied_files:
            shutil.copy2(candidate_dir / relative, directory / relative)
        VARIANT_DIRS[name] = directory
if COMPARISON_MODE == 'reviewer_control':
    controls['optional_analyzer'] = {'sub_agents/patch_reviewer.yaml'}
    candidate_manifest = tool_cli('validate', '--submission-dir', candidate_dir)
    if candidate_manifest['agents'] != 3 or set(candidate_manifest['agent_tool_names']) != {'code_analyzer_agent', 'patch_reviewer_agent'}:
        raise ValueError('Reviewer comparison requires the coder, analyzer and patch reviewer')
    spec = importlib.util.spec_from_file_location('notebook_reviewer_tools', TOOL_SCRIPT)
    reviewer_curator = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(reviewer_curator)
    reviewer_loader = reviewer_curator.SubmissionValidator(candidate_dir)
    reviewer_config = reviewer_loader.load_yaml(candidate_dir / 'sub_agents/patch_reviewer.yaml')
    if reviewer_config.get('tools') != ['read_file'] or reviewer_config.get('include_contents') != 'none':
        raise ValueError('Patch reviewer must have only read_file and no full history')
if COMPARISON_MODE == 'verification_control':
    controls['optional_locator'] = {'sub_agents/test_verifier.yaml'}
    controls['optional_verifier'] = {'sub_agents/repository_locator.yaml'}
    candidate_manifest = tool_cli('validate', '--submission-dir', candidate_dir)
    if candidate_manifest['agents'] != 3 or set(candidate_manifest['agent_tool_names']) != {'repository_locator_agent', 'test_verifier_agent'}:
        raise ValueError('Verification comparison requires the coder, repository locator and test verifier')
    spec = importlib.util.spec_from_file_location('notebook_verification_tools', TOOL_SCRIPT)
    verification_curator = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(verification_curator)
    verification_loader = verification_curator.SubmissionValidator(candidate_dir)
    coder_config = verification_loader.load_yaml(candidate_dir / 'agent.yaml')
    expected_coder_tools = {'run_command', 'read_file', 'edit_file', 'write_file', 'get_status', 'submit_patch'}
    if {tool for tool in coder_config.get('tools', []) if isinstance(tool, str)} != expected_coder_tools:
        raise ValueError('Verification comparison requires the six direct coder tools')
    for helper_path in ['sub_agents/repository_locator.yaml', 'sub_agents/test_verifier.yaml']:
        helper_config = verification_loader.load_yaml(candidate_dir / helper_path)
        if helper_config.get('tools') != ['run_command', 'read_file'] or helper_config.get('include_contents') != 'none':
            raise ValueError('Locator/verifier helpers must have only run_command and read_file and no full history')
for control_variant, removed_paths in controls.items():
    if control_variant not in BENCHMARK_VARIANTS:
        continue
    control_source = RUN_DIR / (control_variant + '_control_source')
    shutil.copytree(candidate_dir, control_source)
    control_yaml = control_source / 'agent.yaml'
    control_yaml.write_text(remove_agent_tools(control_yaml.read_text(encoding='utf-8'), removed_paths), encoding='utf-8')
    # Resolve the changed config before validation to stage only reachable files.
    # This retains coder adapters and excludes detached helper-only assets.
    import importlib.util
    spec = importlib.util.spec_from_file_location('notebook_submission_tools', TOOL_SCRIPT)
    curator = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(curator)
    closure = curator.SubmissionValidator(control_source)
    closure.agent(closure.load_yaml(control_yaml), control_yaml)
    control_eval = control_source / 'eval_config.yaml'
    if control_eval.is_file():
        closure.load_yaml(control_eval)
    control_stage = RUN_DIR / (control_variant + '_control_curated')
    control_stage.mkdir()
    for file in sorted(closure.files):
        target = control_stage / file.relative_to(closure.root)
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(file, target)
    control_zip = RUN_DIR / (control_variant + '_source.zip')
    tool_cli('build', '--submission-dir', control_stage, '--output', control_zip)
    control_dir = RUN_DIR / 'variants' / control_variant
    extract_curated(control_zip, control_dir)
    VARIANT_DIRS[control_variant] = control_dir

VARIANT_MANIFESTS = {name: tool_cli('validate', '--submission-dir', VARIANT_DIRS[name])
                     for name in BENCHMARK_VARIANTS}
budgets = [manifest['evaluation'] for manifest in VARIANT_MANIFESTS.values()]
comparable_budgets = [{key: value for key, value in budget.items() if key != 'timeout_seconds'}
                      for budget in budgets] if COMPARISON_MODE == 'runtime_comparison' else budgets
if any(budget != comparable_budgets[0] for budget in comparable_budgets):
    raise ValueError('Benchmark variants must use identical evaluation budgets')
if COMPARISON_MODE == 'prompt_comparison':
    # Check both source variants even when BENCHMARK_VARIANTS selects only one.
    comparison_manifests = {name: tool_cli('validate', '--submission-dir', VARIANT_DIRS[name])
                            for name in ['baseline', 'candidate']}
    if any(manifest['agents'] != 2 for manifest in comparison_manifests.values()):
        raise ValueError('This prompt comparison requires the same two-agent system in both variants')
    if any(manifest['adapters'] for manifest in comparison_manifests.values()):
        raise ValueError('This prompt comparison uses unchanged base weights. Compare adapter changes in a separate mounted-weight experiment.')
    if comparison_manifests['baseline']['model'] != comparison_manifests['candidate']['model']:
        raise ValueError('Baseline and candidate must use the same base model')
    if comparison_manifests['baseline']['evaluation'] != comparison_manifests['candidate']['evaluation']:
        raise ValueError('Baseline and candidate must use identical evaluation budgets')
    spec = importlib.util.spec_from_file_location('notebook_comparison_tools', TOOL_SCRIPT)
    curator = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(curator)

    def without_instruction(value):
        if isinstance(value, dict):
            return {key: without_instruction(item) for key, item in value.items() if key != 'instruction'}
        if isinstance(value, list):
            return [without_instruction(item) for item in value]
        return value

    def resolved_configuration(name):
        directory = VARIANT_DIRS[name]
        loader = curator.SubmissionValidator(directory)
        return {relative: without_instruction(loader.load_yaml(directory / relative))
                for relative in comparison_manifests[name]['package_files']
                if Path(relative).suffix.lower() in {'.yaml', '.yml'}}

    if resolved_configuration('baseline') != resolved_configuration('candidate'):
        raise ValueError('Prompt comparison requires matching YAML topology, sampling and budget settings; only instruction text may differ')
print('Run:', RUN_DIR)
print('Comparison mode:', COMPARISON_MODE)
print('Variants:', BENCHMARK_VARIANTS)
print('Budgets:', {name: manifest['evaluation'] for name, manifest in VARIANT_MANIFESTS.items()})
print('Sandbox:', SANDBOX, '(Docker scoring parity requires a separate Docker run)' if SANDBOX == 'subprocess' else '')


## 4. Preflight data coverage and choose the shared task set

A task requires its snapshot and matching task-named or commit-named graph and embedding. Automatic selection uses a stable repository round robin; explicit IDs fail early if resources are incomplete. Missing local resources are not a performance result.

In [ ]:
from swegemma.models import load_tasks

TASKS_PATH = DATA_DIR / 'tasks.jsonl'
for required in [TASKS_PATH, DATA_DIR / 'wheels', DATA_DIR / 'sandbox' / 'setup.py', MODEL_PATH]:
    if not required.exists():
        raise FileNotFoundError(f'Missing required dataset/model resource: {required}')
if not any((DATA_DIR / 'wheels').glob('*.whl')):
    raise ValueError('Task dependency wheels directory is empty')
tasks = load_tasks(TASKS_PATH)
SELECTED_TASKS, COVERAGE = select_tasks(tasks, DATA_DIR, TASK_IDS, TASK_LIMIT)
GRAPH_DIR = str(DATA_DIR / 'graphs')
EMBEDDINGS_DIR = str(DATA_DIR / 'embeddings')
selected_ids = [task.instance_id for task in SELECTED_TASKS]
(RUN_DIR / 'task_selection.json').write_text(json.dumps({
    'task_ids': selected_ids, 'selection': 'explicit' if TASK_IDS else 'stable_round_robin_by_repo',
    'available_tasks': len(tasks), 'complete_tasks': sum(all(row.values()) for row in COVERAGE.values()),
    'resources': {identifier: COVERAGE[identifier] for identifier in selected_ids},
}, indent=2), encoding='utf-8')
print(f'Selected {len(SELECTED_TASKS)} identical tasks for each variant:', selected_ids)
if len(SELECTED_TASKS) <= 2:
    print('This is a smoke test, not reliable evidence of general performance.')


## 5. Start the required Gemma model locally

Uses the starter vLLM API, scoring context length 32,768, GPU memory fraction 0.80, and LoRA only when a manifest exists. `EMPTY` is a local routing placeholder. No external API is called.

In [ ]:
import litellm
import torch
from adk_submission import VllmConfig, VllmServer, discover_adapters
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS
from swegemma.models.discovery import validate_single_declared_model

litellm.drop_params = True
TARGET_MODEL_NAME = 'gemma-4-31b-it-qat-w4a16-ct'
INFERENCE_API_KEY = 'EMPTY'  # Local vLLM routing placeholder, no external API.
declared_model = validate_single_declared_model(candidate_dir)
if declared_model != TARGET_MODEL_NAME:
    raise ValueError('Use the competition-required Gemma base model')
adapters = discover_adapters(str(candidate_dir), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 0
if not gpu_count:
    raise RuntimeError('Enable Kaggle GPU acceleration before starting the model')
tp_size = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)
vllm_cfg = VllmConfig(
    model=str(MODEL_PATH), port=8000, host='127.0.0.1',
    tool_call_parser='gemma4', reasoning_parser='gemma4', max_model_len=32768,
    dtype='bfloat16' if torch.cuda.is_bf16_supported() else 'auto',
    gpu_memory_utilization=0.80, enable_auto_tool_choice=True,
    enable_lora=bool(adapters), max_loras=8, max_lora_rank=128,
    tensor_parallel_size=tp_size, startup_timeout=60 * 20,
)
server_instance = VllmServer(vllm_cfg, adapter_manifest=adapters)
server_instance.start()
models = server_instance.create_model_registry(
    aliases=[declared_model, TARGET_MODEL_NAME], model_prefix='openai/', api_key=INFERENCE_API_KEY,
)
print(f'Local vLLM started: {server_instance.base_url}; tensor parallel size={tp_size}')


## 6. Generate patches, verify and report

All variants get the same tasks and agent allowances. Runtime comparisons intentionally vary command/grading timeout and thinking; other comparisons retain identical budgets. Subprocess isolation matches the starter; Docker is the closer scoring environment when available. Variants run in the configured order, so repeat/reverse the order when checking timing.

The notebook writes `notebook_task_results.jsonl` and leaves harness `task_results.jsonl`, traces and logs intact. Each variant receives JSON, CSV and Markdown diagnostics. Absent agent time, request usage or context metrics stay unknown; total task duration is not agent-loop time.

In [ ]:
import asyncio
import concurrent.futures
import dataclasses
import importlib.metadata
import time
import traceback
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from swegemma.config import EvalConfig, build_submission_limits
from swegemma.evaluate import Evaluator

def run_sync(coro_or_fn, *args, **kwargs):
    fn = (lambda: coro_or_fn(*args, **kwargs)) if callable(coro_or_fn) else (lambda: coro_or_fn)
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: asyncio.run(fn())).result()
    return asyncio.run(fn())

def serializable_result(result):
    try:
        if isinstance(result, dict):
            value = result
        elif hasattr(result, 'model_dump'):
            value = result.model_dump(mode='json')
        elif dataclasses.is_dataclass(result):
            value = dataclasses.asdict(result)
        elif hasattr(result, '__dict__'):
            value = vars(result)
        else:
            return None
        json.dumps(value)
        return value
    except (TypeError, ValueError):
        return None

def result_field(result, name):
    return result.get(name) if isinstance(result, dict) else getattr(result, name, None)

def runtime_versions():
    versions = {}
    for package in ['swegemma', 'adk-submission', 'adk-eval-core', 'google-adk', 'vllm', 'litellm', 'torch']:
        try:
            versions[package] = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError:
            versions[package] = None
    return versions

versions = runtime_versions()
SUMMARY_ROWS = []
for variant in BENCHMARK_VARIANTS:
    agent_dir = VARIANT_DIRS[variant]
    results_dir = RUN_DIR / 'results' / variant
    results_dir.mkdir(parents=True, exist_ok=False)
    for directory in ['logs', 'patches', 'raw_results']:
        (results_dir / directory).mkdir(exist_ok=True)
    budget = VARIANT_MANIFESTS[variant]['evaluation']
    limits, gen_constraints = build_submission_limits()
    variant_adapters = discover_adapters(str(agent_dir), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
    config = EvalConfig(
        tasks_path=TASKS_PATH, snapshots_dir=DATA_DIR / 'snapshots',
        results_dir=results_dir, submission_dir=agent_dir, models=models, sandbox=SANDBOX,
        timeout_seconds=int(budget.get('timeout_seconds', 300)),
        max_time_minutes=float(budget.get('max_time_minutes', 60)),
        max_tool_calls=int(budget.get('max_tool_calls', 100)),
        max_turns=int(budget['max_turns']) if budget.get('max_turns') is not None else None,
        limits=limits, generation_constraints=gen_constraints, adapter_manifest=variant_adapters,
        context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
        events_compaction_config=EventsCompactionConfig(
            compaction_interval=5, overlap_size=2, token_threshold=14336, event_retention_size=5,
        ),
        graph_dir=GRAPH_DIR, embeddings_dir=EMBEDDINGS_DIR, wheels_dir=DATA_DIR / 'wheels', verbose=True,
    )
    file_hashes = {name: file_sha256(agent_dir / name)
                   for name in VARIANT_MANIFESTS[variant]['package_files']}
    (results_dir / 'run_manifest.json').write_text(json.dumps({
        'variant': variant, 'task_ids': selected_ids, 'budgets': budget, 'sandbox': SANDBOX,
        'file_sha256': file_hashes, 'runtime_versions': versions, 'base_model': TARGET_MODEL_NAME,
        'model_path': str(MODEL_PATH), 'max_model_len': 32768, 'gpu_memory_utilization': 0.80,
        'compaction_interval': 5, 'benchmark_order': BENCHMARK_VARIANTS,
        'embedded_source_sha256': EMBEDDED_SOURCE_HASHES,
        'embedded_baseline_sha256': EMBEDDED_BASELINE_HASHES,
        'candidate_source_label': CANDIDATE_LABEL, 'baseline_source_label': BASELINE_LABEL,
        'comparison_mode': COMPARISON_MODE,
    }, indent=2), encoding='utf-8')
    evaluator = Evaluator(config)
    records = []
    normalized_path = results_dir / 'notebook_task_results.jsonl'
    for idx, task in enumerate(SELECTED_TASKS, start=1):
        print(f'[{variant} {idx}/{len(SELECTED_TASKS)}] {task.instance_id} ({task.repo})')
        identifier = task.instance_id
        if not identifier or any(c in identifier for c in '/\\') or identifier in {'.', '..'}:
            raise ValueError('Unsafe task identifier')
        handler = logging.FileHandler(results_dir / 'logs' / f'observer_{identifier}.log', encoding='utf-8')
        handler.setFormatter(logging.Formatter('%(asctime)s %(levelname)s %(name)s %(message)s'))
        root_logger = logging.getLogger()
        old_level = root_logger.level
        root_logger.setLevel(logging.INFO)
        root_logger.addHandler(handler)
        started = time.monotonic()
        record = {'instance_id': identifier, 'repo': task.repo, 'resolved': None,
                  'test_exit_code': None, 'agent_patch': None, 'tool_calls': None,
                  'duration_seconds': None, 'agent_elapsed_seconds': None, 'error': None}
        try:
            result = run_sync(evaluator.evaluate_task, task=task, task_index=idx, total_tasks=len(SELECTED_TASKS))
            for field in ['resolved', 'test_exit_code', 'agent_patch', 'tool_calls', 'duration_seconds',
                          'agent_elapsed_seconds', 'error', 'error_type', 'stop_reason', 'exit_status', 'stage']:
                value = result_field(result, field)
                if value is not None:
                    record[field] = value
            raw = serializable_result(result)
            if raw is not None:
                record['raw_result'] = raw
                (results_dir / 'raw_results' / f'{identifier}.json').write_text(json.dumps(raw, indent=2), encoding='utf-8')
            patch = record['agent_patch']
            if isinstance(patch, str):
                record['patch_chars'] = len(patch)
                patch_path = results_dir / 'patches' / f'{identifier}.patch'
                if not patch_path.exists():
                    patch_path.write_text(patch, encoding='utf-8')
        except Exception as error:
            record.update(error=str(error), error_type=type(error).__name__)
            (results_dir / 'logs' / f'exception_{identifier}.log').write_text(traceback.format_exc(), encoding='utf-8')
            logging.exception('Evaluation failed for %s', identifier)
        finally:
            record['observer_wall_seconds'] = time.monotonic() - started
            root_logger.removeHandler(handler)
            handler.close()
            root_logger.setLevel(old_level)
        with normalized_path.open('a', encoding='utf-8') as output:
            output.write(json.dumps(record) + '\n')
        records.append(record)
        print({key: record.get(key) for key in ['resolved', 'test_exit_code', 'tool_calls', 'duration_seconds', 'error']})
    report_summary = tool_cli('report', '--results-dir', results_dir,
                              '--output-dir', results_dir / 'report', '--submission-dir', agent_dir)
    SUMMARY_ROWS.append({'variant': variant, 'selected_tasks': len(selected_ids), **report_summary})
(RUN_DIR / 'comparison.json').write_text(json.dumps(SUMMARY_ROWS, indent=2), encoding='utf-8')
try:
    import pandas as pd
    display(pd.DataFrame(SUMMARY_ROWS))
except ImportError:
    print(json.dumps(SUMMARY_ROWS, indent=2))
print('Full logs, patches, traces when emitted by the runtime, and reports:', RUN_DIR)


## 7. Build the chosen release

Inspect reports first. `RELEASE_VARIANT` chooses which evaluated directory becomes `/kaggle/working/submission.zip`. The ZIP contains only active agent assets. Packaging is not proof of task success.

In [ ]:
# Only the selected variant's active dependency closure goes into the ZIP.
# Reports, notebook code, tasks and traces remain in RUN_DIR.
release_dir = VARIANT_DIRS[RELEASE_VARIANT]
tool_cli('validate', '--submission-dir', release_dir)
zip_path = WORKING_DIR / 'submission.zip'
if zip_path.exists():
    # Preserve a previous release before replacing this conventional Kaggle output.
    shutil.copy2(zip_path, RUN_DIR / 'previous_submission.zip')
package_manifest = tool_cli('build', '--submission-dir', release_dir, '--output', zip_path)
(RUN_DIR / 'release_manifest.json').write_text(json.dumps(package_manifest, indent=2), encoding='utf-8')
print('Release variant:', RELEASE_VARIANT)
print('Created:', zip_path, 'bytes:', zip_path.stat().st_size)


## 8. Stop the server when finished

Run after all evaluation or further experiments are complete.

In [ ]:
server_instance.stop()
print('Local model server stopped.')
